In [11]:
import pandas as pd

pd.reset_option("display.max_rows")
pd.reset_option("display.max_columns")


In [3]:
import pandas as pd

fantasy = pd.read_csv('../data/master/master_fantasy.csv')
fantasy

,match_id,nick_name,full_name,pos,team,cost,mins,goals,assists,cs,goals_c,own_goals,y_c,r_c,bonus,points
0,918893,Cech,Petr Cech,Goalkeeper,ARS,0.0,90,0,0,0,3,0,0,0,0,0
1,918893,Walcott,Theo Walcott,Midfielder,ARS,0.0,21,0,0,0,0,0,0,0,0,0
2,918893,Ozil,Mesut Ozil,Midfielder,ARS,0.0,90,0,0,0,3,0,0,0,0,0
3,918893,Monreal,Nacho Monreal,Defender,ARS,0.0,90,0,0,0,3,0,0,0,0,0
4,918893,Ramsey,Aaron Ramsey,Midfielder,ARS,0.0,29,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86711,2444849,Ait Nouri,Rayan Ait Nouri,Defender,WOL,6.0,65,0,0,0,1,0,0,0,0,2
86712,2444849,Rodrigo Gomes,Rodrigo Martins Gomes,Midfielder,WOL,4.5,30,0,0,0,0,0,0,0,0,1
86713,2444849,André,André Trindade da Costa Neto,Midfielder,WOL,5.5,90,0,0,0,1,0,0,0,0,2
86714,2444849,Toti,Toti António Gomes,Defender,WOL,4.5,90,0,0,0,1,0,0,0,0,2


In [4]:
fantasy['pos'].unique()

array(['Goalkeeper', 'Midfielder', 'Defender', 'Forward'], dtype=object)

In [5]:
master_players = (
    fantasy.groupby(["team", "full_name"])["nick_name"]
    .agg(lambda x: (
        x.unique()[0] if len(x.unique()) == 1 else sorted(x.unique().tolist())
    ))
    .reset_index()
)

master_players

,team,full_name,nick_name
0,ARS,Aaron Ramsdale,Ramsdale
1,ARS,Aaron Ramsey,Ramsey
2,ARS,Ainsley Maitland-Niles,Maitland-Niles
3,ARS,Albert Sambi Lokonga,Sambi Lokonga
4,ARS,Alex Iwobi,Iwobi
...,...,...,...
1991,WOL,Vitor Machado Ferreira,Vitor Ferreira
1992,WOL,Will Norris,Norris
1993,WOL,Willian José Da Silva,Willian José
1994,WOL,Willy Boly,Boly


In [6]:
master_players['team'].unique()

array(['ARS', 'AVL', 'BHA', 'BOU', 'BRE', 'BUR', 'CAR', 'CHE', 'CRY',
       'EVE', 'FUL', 'HUD', 'IPS', 'LEE', 'LEI', 'LIV', 'LUT', 'MCI',
       'MUN', 'NEW', 'NFO', 'NOR', 'SHU', 'SOU', 'STK', 'SWA', 'TOT',
       'WAT', 'WBA', 'WHU', 'WOL'], dtype=object)

In [12]:
master_players[master_players['team'] == 'BHA']

,team,full_name,nick_name
150,BHA,Aaron Connolly,Connolly
151,BHA,Aaron Mooy,Mooy
152,BHA,Adam Lallana,Lallana
153,BHA,Adam Webster,"[Webster, Webster (Adam)]"
154,BHA,Alexis Mac Allister,Mac Allister
...,...,...,...
235,BHA,Uwe Hünemeier,Hünemeier
236,BHA,Valentín Barco,Barco
237,BHA,Yankuba Minteh,Minteh
238,BHA,Yasin Ayari,Ayari


Joseph Willock	Willock (Joseph)
38	ARS	Joseph Willock	Willock

Reiss Nelson	Nelson
66	ARS	Reiss Nelson	Nelson (Reiss)

Calum Chambers	Chambers
92	AVL	Calum Chambers	Chambers (Calum)

Carney Chukwuemeka	Chukwuemeka
95	AVL	Carney Chukwuemeka	Chukwuemeka (Carney)

Keinan Davis	Davis
123	AVL	Keinan Davis	Davis (Keinan)

Leander Dendoncker	Dendoncker (Leander)
128	AVL	Leander Dendoncker	Dendoncker

In [13]:
import cloudscraper
from lxml import html
from premier_league.transfers.transfers import Transfers as BaseTransfers
import pandas as pd

class TransfersCF(BaseTransfers):
    def request_url_page(self):
        """Override to use cloudscraper instead of requests."""
        scraper = cloudscraper.create_scraper()  # bypass Cloudflare
        r = scraper.get(self.url)
        r.raise_for_status()
        return html.fromstring(r.content)

seasons = ["2017-2018", "2018-2019", "2019-2020", "2020-2021", 
           "2021-2022", "2022-2023", "2023-2024", "2024-2025", "2025-2026"]

main_df = pd.DataFrame(columns=['Date','Name','Pos','From','To','Season'])

def clean_in_table(df, team, season):
    """Normalize a raw scraped IN table into the standard schema."""
    if df.empty:
        return pd.DataFrame(columns=['Date','Name','Pos','From','To','Season'])
    
    # Drop header row if present
    if "Date" in df.iloc[0].values:
        df = df.iloc[1:]
    
    # Ensure at least 4 cols
    if df.shape[1] < 4:
        return pd.DataFrame(columns=['Date','Name','Pos','From','To','Season'])
    
    df = df.iloc[:, :4]  # only take first 4 cols
    df.columns = ['Date','Name','Pos','From']  # rename consistently
    df['To'] = team
    df['Season'] = season
    
    return df[['Date','Name','Pos','From','To','Season']]

for season in seasons:
    transfers = TransfersCF(target_season=season, league="Premier League", cache=False)
    print(f"Season: {season}")
    teams = transfers.get_all_current_teams()
    for team in teams:
        if "Competition News" in team:
            continue
        print(f"Fetching transfers IN for {team} in {season}")
        in_df = pd.DataFrame(transfers.transfer_in_table(team))
        in_df = clean_in_table(in_df, team, season)
        main_df = pd.concat([main_df, in_df], ignore_index=True)

print(main_df.head(20))




Season: 2017-2018
Fetching transfers IN for Afc Bournemouth in 2017-2018
Fetching transfers IN for Arsenal FC in 2017-2018
Fetching transfers IN for Brighton & Hove Albion in 2017-2018
Fetching transfers IN for Burnley FC in 2017-2018
Fetching transfers IN for Chelsea FC in 2017-2018
Fetching transfers IN for Crystal Palace in 2017-2018
Fetching transfers IN for Everton FC in 2017-2018
Fetching transfers IN for Huddersfield Town in 2017-2018
Fetching transfers IN for Leicester City in 2017-2018
Fetching transfers IN for Liverpool FC in 2017-2018
Fetching transfers IN for Manchester City in 2017-2018
Fetching transfers IN for Manchester United in 2017-2018
Fetching transfers IN for Newcastle United in 2017-2018
Fetching transfers IN for Southampton FC in 2017-2018
Fetching transfers IN for Stoke City in 2017-2018
Fetching transfers IN for Swansea City in 2017-2018
Fetching transfers IN for Tottenham Hotspur in 2017-2018
Fetching transfers IN for Watford FC in 2017-2018
Fetching transfer

In [14]:
main_df

,Date,Name,Pos,From,To,Season
0,01/18,Baily Cargill,DF,Fleetwood Town,Afc Bournemouth,2017-2018
1,01/18,Ryan Allsop,GK,Blackpool FC,Afc Bournemouth,2017-2018
2,07/17,Nathan Aké,DF,Chelsea FC,Afc Bournemouth,2017-2018
3,07/17,Asmir Begović,GK,Chelsea FC,Afc Bournemouth,2017-2018
4,07/17,Matt Butcher,DF,Yeovil Town,Afc Bournemouth,2017-2018
...,...,...,...,...,...,...
2245,07/25,Tawanda Chirewa,MF,Huddersfield Town,Wolverhampton Wanderers,2025-2026
2246,07/25,Fábio Silva,FW,UD Las Palmas,Wolverhampton Wanderers,2025-2026
2247,07/25,Fer López,MF,RC Celta,Wolverhampton Wanderers,2025-2026
2248,07/25,Joe Hodge,MF,Huddersfield Town,Wolverhampton Wanderers,2025-2026


In [15]:
main_df['To'].unique()

array(['Afc Bournemouth', 'Arsenal FC', 'Brighton & Hove Albion',
       'Burnley FC', 'Chelsea FC', 'Crystal Palace', 'Everton FC',
       'Huddersfield Town', 'Leicester City', 'Liverpool FC',
       'Manchester City', 'Manchester United', 'Newcastle United',
       'Southampton FC', 'Stoke City', 'Swansea City',
       'Tottenham Hotspur', 'Watford FC', 'West Bromwich Albion',
       'West Ham United', 'Cardiff City', 'Fulham FC',
       'Wolverhampton Wanderers', 'Aston Villa', 'Norwich City',
       'Sheffield United', 'Leeds United', 'Brentford FC',
       'Nottingham Forest', 'Luton Town', 'Ipswich Town',
       'Sunderland Afc'], dtype=object)

In [16]:
club_mapping = {
    'Afc Bournemouth': 'BOU',
    'Arsenal FC': 'ARS',
    'Brighton & Hove Albion': 'BHA',
    'Burnley FC': 'BUR',
    'Chelsea FC': 'CHE',
    'Crystal Palace': 'CRY',
    'Everton FC': 'EVE',
    'Huddersfield Town': 'HUD',
    'Leicester City': 'LEI',
    'Liverpool FC': 'LIV',
    'Manchester City': 'MCI',
    'Manchester United': 'MUN',
    'Newcastle United': 'NEW',
    'Southampton FC': 'SOU',
    'Stoke City': 'STK',
    'Swansea City': 'SWA',
    'Tottenham Hotspur': 'TOT',
    'Watford FC': 'WAT',
    'West Bromwich Albion': 'WBA',
    'West Ham United': 'WHU',
    'Cardiff City': 'CAR',
    'Fulham FC': 'FUL',
    'Wolverhampton Wanderers': 'WOL',
    'Aston Villa': 'AVL',
    'Norwich City': 'NOR',
    'Sheffield United': 'SHU',
    'Leeds United': 'LEE',
    'Brentford FC': 'BRE',
    'Nottingham Forest': 'NFO',
    'Luton Town': 'LUT',
    'Ipswich Town': 'IPS',
    'Sunderland Afc': 'SUN'
    }

position_mapping = {
    'GK': 'Goalkeeper',
    'DF': 'Defender',
    'MF': 'Midfielder',
    'FW': 'Forward'
}



In [18]:
for index, row in main_df.iterrows():
    team = row['To']
    position = row['Pos']
    main_df.at[index, 'To'] = club_mapping[team]
    main_df.at[index, 'Pos'] = position_mapping[position]

main_df
    

,Date,Name,Pos,From,To,Season
0,01/18,Baily Cargill,Defender,Fleetwood Town,BOU,2017-2018
1,01/18,Ryan Allsop,Goalkeeper,Blackpool FC,BOU,2017-2018
2,07/17,Nathan Aké,Defender,Chelsea FC,BOU,2017-2018
3,07/17,Asmir Begović,Goalkeeper,Chelsea FC,BOU,2017-2018
4,07/17,Matt Butcher,Defender,Yeovil Town,BOU,2017-2018
...,...,...,...,...,...,...
2245,07/25,Tawanda Chirewa,Midfielder,Huddersfield Town,WOL,2025-2026
2246,07/25,Fábio Silva,Forward,UD Las Palmas,WOL,2025-2026
2247,07/25,Fer López,Midfielder,RC Celta,WOL,2025-2026
2248,07/25,Joe Hodge,Midfielder,Huddersfield Town,WOL,2025-2026


In [20]:
# Convert
main_df["Date"] = pd.to_datetime(main_df["Date"], format="%m/%y")  
main_df["Date"] = main_df["Date"].dt.to_period("M").dt.to_timestamp()  # normalize to first of month
main_df

,Date,Name,Pos,From,To,Season
0,2018-01-01,Baily Cargill,Defender,Fleetwood Town,BOU,2017-2018
1,2018-01-01,Ryan Allsop,Goalkeeper,Blackpool FC,BOU,2017-2018
2,2017-07-01,Nathan Aké,Defender,Chelsea FC,BOU,2017-2018
3,2017-07-01,Asmir Begović,Goalkeeper,Chelsea FC,BOU,2017-2018
4,2017-07-01,Matt Butcher,Defender,Yeovil Town,BOU,2017-2018
...,...,...,...,...,...,...
2245,2025-07-01,Tawanda Chirewa,Midfielder,Huddersfield Town,WOL,2025-2026
2246,2025-07-01,Fábio Silva,Forward,UD Las Palmas,WOL,2025-2026
2247,2025-07-01,Fer López,Midfielder,RC Celta,WOL,2025-2026
2248,2025-07-01,Joe Hodge,Midfielder,Huddersfield Town,WOL,2025-2026
